# The second best model we'll do
* bangbangbang

## O. Setup 

In [0]:
NOM_EQUIPE = "telecacaton"   # ← remplacez par le nom de votre équipe

# Ne touchez pas au reste
TABLE_PREDICTIONS = f"workspace.default.predictions_equipe_{NOM_EQUIPE}"
print(f"Votre table de prédictions : {TABLE_PREDICTIONS}")

In [0]:
#%sql GRANT MODIFY ON TABLE workspace.default.#predictions_equipe_telecacaton TO `cyprien.mas@telecom-paris.fr`;
#GRANT MODIFY ON TABLE workspace.default.#predictions_equipe_telecacaton TO `hugo.hennion@telecom-paris.fr`;
#GRANT MODIFY ON TABLE workspace.default.#predictions_equipe_telecacaton TO `clement.pesquet@telecom-paris.#fr`;

## 1. Model

### Import et Setup

In [0]:
%pip install lightgbm
%pip install xgboost
#dbutils.library.restartPython()

In [0]:
%pip install optuna


In [0]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from pyspark.sql import functions as F
from pyspark.sql.types import LongType
import warnings, gc, time

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:.4f}".format

SEED = 42
np.random.seed(SEED)
print("✅ Imports OK")

### Chargement des données

In [0]:
train_df = spark.table("workspace.default.histo_ventes_train")
test_df  = spark.table("workspace.default.histo_ventes_test")

cols_utiles = ["semaine", "code_agence", "code_article", "quantite"]

print("⏳ Chargement train...")
train_pd = train_df.select(cols_utiles).toPandas()
print(f"✅ Train : {len(train_pd):,} lignes")

print("⏳ Chargement test...")
test_pd = test_df.select(["semaine", "code_agence", "code_article"]).toPandas()
print(f"✅ Test  : {len(test_pd):,} lignes")

def parse_semaine(df):
    df = df.copy()
    df["annee"]   = df["semaine"].str.split("-").str[0].astype(int)
    df["num_sem"] = df["semaine"].str.split("-").str[1].astype(int)
    df["week_id"] = df["annee"] * 100 + df["num_sem"]
    return df

train_pd = parse_semaine(train_pd)
test_pd  = parse_semaine(test_pd)

train_pd["is_test"] = 0
test_pd["is_test"]  = 1
test_pd["quantite"] = np.nan

all_df = pd.concat([train_pd, test_pd], ignore_index=True)
all_df = all_df.sort_values(["code_agence", "code_article", "week_id"]).reset_index(drop=True)

print(f"Total : {len(all_df):,} lignes | {all_df.memory_usage(deep=True).sum()/1e6:.0f} MB")

In [0]:
n_pairs   = all_df[["code_agence", "code_article"]].drop_duplicates().shape[0]
n_weeks   = all_df["week_id"].nunique()
size_gb   = n_pairs * n_weeks * 8 / 1e9  # float64

print(f"Paires uniques  : {n_pairs:,}")
print(f"Semaines uniques: {n_weeks:,}")
print(f"Taille pivot    : {size_gb:.1f} GB")
print(f"Taille all_df   : {all_df.memory_usage(deep=True).sum()/1e9:.2f} GB")

### 2. Feature Engineering

In [0]:
def add_features_stable(df):
    df         = df.copy()
    grp        = ["code_agence", "code_article"]
    train_only = df[df["is_test"] == 0]

    df["_pair_key"] = (
        df["code_agence"].astype(str) + "_" + df["code_article"].astype(str)
    )

    # ── Pivot unique (25 MB — tient largement en RAM) ─────────────────────
    print("  → Pivot...")
    pivot = (
        df[["week_id", "_pair_key", "quantite"]]
        .pivot_table(index="week_id", columns="_pair_key",
                     values="quantite", aggfunc="first")
        .sort_index()
        .astype("float32")
    )
    pivot_s1   = pivot.shift(1)
    pivot_zero = (pivot == 0).astype("float32").shift(1)

    # ── Merge helper : dépivot + jointure en une passe ────────────────────
    # Clé : on merge sur (week_id, _pair_key) via stack() — plus rapide
    # que melt() sur de larges DataFrames
    def add_pivot_feature(df_main, pivot_feat, feat_name):
        s = (
            pivot_feat
            .stack(dropna=False)               # → Series multi-index (week_id, pair_key)
            .rename(feat_name)
            .reset_index()
            .rename(columns={"_pair_key": "_pair_key"})
        )
        return df_main.merge(s, on=["week_id", "_pair_key"], how="left")

    # ── Lags ──────────────────────────────────────────────────────────────
    print("  → Lags...")
    for lag in [1, 2, 3, 4, 5, 6, 8, 13, 26, 52, 104]:
        df = add_pivot_feature(df, pivot.shift(lag).astype("float32"), f"lag_{lag}")

    # ── Rolling mean & std (pas de median → trop lent) ────────────────────
    print("  → Rolling mean / std / max...")
    for w in [4, 8, 13, 26, 52]:
        r = pivot_s1.rolling(w, min_periods=1)
        df = add_pivot_feature(df, r.mean().astype("float32"), f"roll_mean_{w}")
        df = add_pivot_feature(df, r.std().astype("float32"),  f"roll_std_{w}")
        df = add_pivot_feature(df, r.max().astype("float32"),  f"roll_max_{w}")
        # PAS de .median() ici — remplacée par ewma ci-dessous
        del r; gc.collect()

    # ── EWMA — remplace avantageusement la médiane rolling ────────────────
    print("  → EWMA...")
    for span in [4, 8, 13, 26, 52]:
        feat = pivot_s1.ewm(span=span, min_periods=1).mean().astype("float32")
        df   = add_pivot_feature(df, feat, f"ewma_{span}")
        del feat; gc.collect()

    # ── Zero rate rolling ─────────────────────────────────────────────────
    print("  → Zero rate rolling...")
    for w in [4, 8, 13, 26, 52]:
        feat = pivot_zero.rolling(w, min_periods=1).mean().astype("float32")
        df   = add_pivot_feature(df, feat, f"zero_rate_{w}")
        del feat; gc.collect()

    # ── Streak zeros — vectorisé NumPy colonne par colonne ────────────────
    # np.apply_along_axis crashait → boucle sur les colonnes à la place
    # 11908 colonnes × 260 lignes = négligeable
    print("  → Streak zeros...")
    vals      = pivot.values  # shape (260, 11908)
    n_weeks, n_pairs = vals.shape
    streak_arr = np.zeros((n_weeks, n_pairs), dtype="float32")

    for j in range(n_pairs):
        col   = vals[:, j]
        count = 0.0
        for i in range(n_weeks):
            prev = col[i - 1] if i > 0 else np.nan
            if np.isnan(prev):
                count = 0.0
            elif prev == 0:
                count += 1.0
            else:
                count = 0.0
            streak_arr[i, j] = count

    streak_pivot = pd.DataFrame(
        streak_arr, index=pivot.index, columns=pivot.columns
    )
    df = add_pivot_feature(df, streak_pivot, "streak_zeros")
    del streak_arr, streak_pivot; gc.collect()

    # ── Tendances ─────────────────────────────────────────────────────────
    print("  → Tendances...")
    for h_short, h_long in [(4, 8), (8, 13), (13, 26), (26, 52)]:
        feat = ((pivot.shift(1) - pivot.shift(h_long + 1)) / h_long).astype("float32")
        df   = add_pivot_feature(df, feat, f"trend_{h_short}_{h_long}")
        del feat; gc.collect()

    # ── Libérer les pivots ────────────────────────────────────────────────
    del pivot, pivot_s1, pivot_zero; gc.collect()

    # ── Stats globales (groupby simple) ───────────────────────────────────
    print("  → Stats globales paire...")
    pair_stats = (
        train_only.groupby(grp)["quantite"]
        .agg(
            pair_mean="mean",   pair_median="median",
            pair_max="max",     pair_std="std",
            pair_count="count",
            pair_q10=lambda x: x.quantile(0.10),
            pair_q90=lambda x: x.quantile(0.90),
        ).reset_index()
    )
    df = df.merge(pair_stats, on=grp, how="left")
    del pair_stats; gc.collect()

    print("  → Stats semaine × paire...")
    sem_stats = (
        train_only.groupby(grp + ["num_sem"])["quantite"]
        .agg(sem_mean="mean", sem_max="max",
             sem_median="median", sem_std="std")
        .reset_index()
    )
    df = df.merge(sem_stats, on=grp + ["num_sem"], how="left")
    del sem_stats; gc.collect()

    print("  → Stats agence / article...")
    agence_g = (
        train_only.groupby("code_agence")["quantite"]
        .agg(agence_mean="mean", agence_median="median", agence_std="std")
        .reset_index()
    )
    article_g = (
        train_only.groupby("code_article")["quantite"]
        .agg(article_mean="mean", article_median="median", article_std="std")
        .reset_index()
    )
    agence_sem = (
        train_only.groupby(["code_agence", "num_sem"])["quantite"]
        .agg(agence_sem_mean="mean", agence_sem_median="median")
        .reset_index()
    )
    article_sem = (
        train_only.groupby(["code_article", "num_sem"])["quantite"]
        .agg(article_sem_mean="mean", article_sem_median="median")
        .reset_index()
    )
    df = df.merge(agence_g,   on="code_agence",               how="left")
    df = df.merge(article_g,  on="code_article",              how="left")
    df = df.merge(agence_sem, on=["code_agence",  "num_sem"], how="left")
    df = df.merge(article_sem,on=["code_article", "num_sem"], how="left")
    del agence_g, article_g, agence_sem, article_sem; gc.collect()

    print("  → Zero rate global & semaines actives...")
    zero_rate_g = (
        train_only.groupby(grp)["quantite"]
        .apply(lambda x: (x == 0).mean())
        .reset_index().rename(columns={"quantite": "zero_rate_global"})
    )
    active_w = (
        train_only.groupby(grp)
        .apply(lambda x: (x["quantite"] > 0).sum())
        .reset_index().rename(columns={0: "n_active_weeks"})
    )
    df = df.merge(zero_rate_g, on=grp, how="left")
    df = df.merge(active_w,    on=grp, how="left")
    del zero_rate_g, active_w; gc.collect()

    # ── Features dérivées ─────────────────────────────────────────────────
    print("  → Features dérivées...")
    df["pct_active"]           = df["n_active_weeks"] / (df["pair_count"]  + 1e-5)
    df["cv_pair"]              = df["pair_std"]       / (df["pair_mean"]   + 1e-5)
    df["ratio_sem_vs_pair"]    = df["sem_mean"]       / (df["pair_mean"]   + 1e-5)
    df["ratio_n1_vs_mean"]     = df["lag_52"]         / (df["pair_mean"]   + 1e-5)
    df["ratio_lag1_vs_mean"]   = df["lag_1"]          / (df["pair_mean"]   + 1e-5)
    df["ratio_ewma4_vs_mean"]  = df["ewma_4"]         / (df["pair_mean"]   + 1e-5)
    df["ratio_ewma13_vs_mean"] = df["ewma_13"]        / (df["pair_mean"]   + 1e-5)
    df["spread_q90_q10"]       = df["pair_q90"]       - df["pair_q10"]
    df["rel_spread"]           = df["spread_q90_q10"] / (df["pair_mean"]   + 1e-5)
    df["momentum_4_13"]        = df["lag_4"]          / (df["lag_13"]      + 1e-5)
    df["momentum_13_52"]       = df["lag_13"]         / (df["lag_52"]      + 1e-5)
    df["momentum_1_4"]         = df["lag_1"]          / (df["lag_4"]       + 1e-5)

    for k in [1, 2, 3, 4]:
        df[f"sin_sem_{k}"] = np.sin(2 * np.pi * k * df["num_sem"] / 52).astype("float32")
        df[f"cos_sem_{k}"] = np.cos(2 * np.pi * k * df["num_sem"] / 52).astype("float32")
    df["mois"]             = ((df["num_sem"] - 1) // 4 + 1).clip(1, 12)
    df["sin_mois"]         = np.sin(2 * np.pi * df["mois"] / 12).astype("float32")
    df["cos_mois"]         = np.cos(2 * np.pi * df["mois"] / 12).astype("float32")

    vacances = [1, 2, 7, 8, 17, 18, 19, 28, 29, 30, 31, 32, 43, 44, 52]
    df["is_vacances"]      = df["num_sem"].isin(vacances).astype("int8")
    df["is_fin_trimestre"] = df["num_sem"].isin([13, 26, 39, 52]).astype("int8")
    df["is_debut_annee"]   = (df["num_sem"] <= 4).astype("int8")
    df["is_fin_annee"]     = (df["num_sem"] >= 49).astype("int8")

    df = df.drop(columns=["_pair_key"])
    print(f"  ✅ Shape finale : {df.shape}")
    return df


print("⏳ Feature engineering stable...")
t0 = time.time()
all_df = add_features_stable(all_df)
print(f"✅ Terminé en {(time.time()-t0)/60:.1f} min")
print(f"   RAM all_df : {all_df.memory_usage(deep=True).sum()/1e9:.2f} GB")

### 3. Split Train / Validation / Test

In [0]:
FEATURES = [c for c in all_df.columns if c not in [
    "semaine", "code_agence", "code_article", "quantite",
    "is_test", "annee", "week_id",
    # Exclure les stats brutes redondantes avec les ratios
]]

# Vérifier qu'aucune feature cible n'a fui
assert "quantite" not in FEATURES
print(f"Nombre de features : {len(FEATURES)}")

# Split :
# - train : tout avant 2024 (modèle de base)
# - val   : 2024 (évaluation honnête, proche du test 2025)
# - retrain : train + val (pour les prédictions finales)
train_mask  = (all_df["is_test"] == 0) & (all_df["semaine"] < "2024-01")
val_mask    = (all_df["is_test"] == 0) & (all_df["semaine"].between("2024-01", "2024-52"))
test_mask   =  all_df["is_test"] == 1

X_train = all_df.loc[train_mask, FEATURES]
y_train = all_df.loc[train_mask, "quantite"]
X_val   = all_df.loc[val_mask,   FEATURES]
y_val   = all_df.loc[val_mask,   "quantite"]
X_test  = all_df.loc[test_mask,  FEATURES]

# Retrain = train + val (pour les prédictions finales sur le test)
retrain_mask = all_df["is_test"] == 0
X_retrain = all_df.loc[retrain_mask, FEATURES]
y_retrain = all_df.loc[retrain_mask, "quantite"]

# Cibles binaires pour le classificateur zéro
y_train_bin   = (y_train == 0).astype(int)
y_val_bin     = (y_val   == 0).astype(int)
y_retrain_bin = (y_retrain == 0).astype(int)

# Masques non-zéros pour le régresseur
nz_train = y_train > 0
nz_val   = y_val   > 0

print(f"X_train : {X_train.shape} | X_val : {X_val.shape} | X_test : {X_test.shape}")
print(f"X_retrain : {X_retrain.shape}")
print(f"Taux de zéros — train : {y_train_bin.mean():.2%} | val : {y_val_bin.mean():.2%}")

# ── Métriques ──────────────────────────────────────────────────────────────
def wape(pred, true):
    return float(np.sum(np.abs(pred - true)) / (np.sum(true) + 1e-10))

def wape_lgb(y_pred, dataset):
    y_true = dataset.get_label()
    return "wape", np.sum(np.abs(y_pred - y_true)) / (np.sum(y_true) + 1e-10), False

### 3.5 Classificateur zero

In [0]:
print("=" * 60)
print("STAGE 1 — Classificateur zéros/non-zéros")
print("=" * 60)

# ── Optuna : tuning classificateur ────────────────────────────────────────
def objective_clf(trial):
    params = {
        "objective":         "binary",
        "metric":            "auc",
        "verbosity":         -1,
        "n_jobs":            -1,
        "seed":              SEED,
        "learning_rate":     trial.suggest_float("lr",  0.02, 0.15, log=True),
        "num_leaves":        trial.suggest_int("nl",    31, 511),
        "min_child_samples": trial.suggest_int("mcs",   5,  50),
        "feature_fraction":  trial.suggest_float("ff",  0.5, 0.9),
        "bagging_fraction":  trial.suggest_float("bf",  0.5, 0.9),
        "bagging_freq":      1,
        "reg_alpha":         trial.suggest_float("ra",  1e-3, 1.0, log=True),
        "reg_lambda":        trial.suggest_float("rl",  1e-3, 5.0, log=True),
    }
    dtrain = lgb.Dataset(X_train, label=y_train_bin)
    dval   = lgb.Dataset(X_val,   label=y_val_bin, reference=dtrain)
    m = lgb.train(
        params, dtrain,
        num_boost_round=800,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)],
    )
    # Optimiser sur F1 zéro (pas AUC) → plus corrélé au WAPE
    proba = m.predict(X_val)
    from sklearn.metrics import f1_score
    best_f1 = max(
        f1_score(y_val_bin, (proba > t).astype(int), zero_division=0)
        for t in np.arange(0.3, 0.8, 0.05)
    )
    return -best_f1

print("⚙️  Optuna — classificateur zéro (50 trials)...")
study_clf = optuna.create_study(direction="minimize", sampler=TPESampler(seed=SEED))
study_clf.optimize(objective_clf, n_trials=50, show_progress_bar=False)
best_clf_params = study_clf.best_params
print(f"✅ Meilleurs params clf : {best_clf_params}")

# ── Entraînement final classificateur ─────────────────────────────────────
clf_params = {
    "objective":         "binary",
    "metric":            "auc",
    "verbosity":         -1,
    "n_jobs":            -1,
    "seed":              SEED,
    "learning_rate":     best_clf_params["lr"],
    "num_leaves":        best_clf_params["nl"],
    "min_child_samples": best_clf_params["mcs"],
    "feature_fraction":  best_clf_params["ff"],
    "bagging_fraction":  best_clf_params["bf"],
    "bagging_freq":      1,
    "reg_alpha":         best_clf_params["ra"],
    "reg_lambda":        best_clf_params["rl"],
}

dtrain_clf = lgb.Dataset(X_train, label=y_train_bin)
dval_clf   = lgb.Dataset(X_val,   label=y_val_bin, reference=dtrain_clf)

clf_model = lgb.train(
    clf_params, dtrain_clf,
    num_boost_round=2000,
    valid_sets=[dval_clf],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)],
)

# Calibrer le seuil sur la validation pour maximiser la réduction de WAPE
clf_proba_val = clf_model.predict(X_val)

best_thresh, best_wape_clf = 0.5, float("inf")
for thresh in np.arange(0.20, 0.85, 0.02):
    pred_zero_mask = clf_proba_val > thresh
    y_pred_clf = y_val.copy() * 0.0  # placeholder
    # On verra le WAPE total après Stage 2 ; ici on stocke juste le seuil optimal
    # Proxy : minimiser les faux non-zéros (erreur absolue sur les vrais zéros)
    false_positives = (~y_val_bin.astype(bool)) & pred_zero_mask
    false_negatives =  y_val_bin.astype(bool)  & (~pred_zero_mask)
    score = false_positives.sum() * 1.0 + false_negatives.sum() * 2.0  # pénalise FN
    if score < best_wape_clf:
        best_wape_clf = score
        best_thresh = thresh

print(f"✅ Seuil zero optimal : {best_thresh:.2f}")

# ── Retrain classificateur sur train+val ───────────────────────────────────
print("⏳ Retrain classificateur sur train+val...")
dtrain_clf_full = lgb.Dataset(X_retrain, label=y_retrain_bin)
clf_model_final = lgb.train(
    {**clf_params, "learning_rate": clf_params["learning_rate"] * 0.8},
    dtrain_clf_full,
    num_boost_round=int(clf_model.best_iteration * 1.1),
    callbacks=[lgb.log_evaluation(-1)],
)
print("✅ Classificateur final prêt")

### 4. Entraînement LightGBM

In [0]:
print("=" * 60)
print("STAGE 2 — Régresseur LightGBM Tweedie (sur non-zéros)")
print("=" * 60)

# ── Optuna : tuning régresseur ────────────────────────────────────────────
def objective_reg(trial):
    params = {
        "objective":               "tweedie",
        "tweedie_variance_power":  trial.suggest_float("tvp", 1.0, 1.9),
        "metric":                  "None",
        "verbosity":               -1,
        "n_jobs":                  -1,
        "seed":                    SEED,
        "learning_rate":           trial.suggest_float("lr",  0.02, 0.15, log=True),
        "num_leaves":              trial.suggest_int("nl",    63, 511),
        "min_child_samples":       trial.suggest_int("mcs",   5,  50),
        "feature_fraction":        trial.suggest_float("ff",  0.5, 0.9),
        "bagging_fraction":        trial.suggest_float("bf",  0.5, 0.9),
        "bagging_freq":            1,
        "reg_alpha":               trial.suggest_float("ra",  1e-4, 1.0, log=True),
        "reg_lambda":              trial.suggest_float("rl",  1e-4, 5.0, log=True),
        "min_gain_to_split":       trial.suggest_float("mgs", 1e-5, 0.01, log=True),
    }
    # Entraîner sur TOUS les points train (pas seulement non-zéros)
    # → le modèle apprend aussi à prédire 0 quand c'est justifié
    dtrain = lgb.Dataset(X_train, label=y_train)
    dval   = lgb.Dataset(X_val,   label=y_val, reference=dtrain)
    m = lgb.train(
        params, dtrain,
        num_boost_round=1500,
        valid_sets=[dval],
        feval=wape_lgb,
        callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(-1)],
    )
    preds = np.clip(np.round(m.predict(X_val)), 0, None)
    return wape(preds, y_val.values)

print("⚙️  Optuna — régresseur Tweedie (80 trials)...")
study_reg = optuna.create_study(direction="minimize", sampler=TPESampler(seed=SEED))
study_reg.optimize(objective_reg, n_trials=80, show_progress_bar=False)
best_reg_params = study_reg.best_params
print(f"✅ Meilleurs params reg : {best_reg_params}")
print(f"   WAPE Optuna val : {study_reg.best_value:.4f}")

# ── Entraînement multi-phases avec les meilleurs params ───────────────────
reg_base = {
    "objective":               "tweedie",
    "tweedie_variance_power":  best_reg_params["tvp"],
    "metric":                  "None",
    "verbosity":               -1,
    "n_jobs":                  -1,
    "seed":                    SEED,
    "num_leaves":              best_reg_params["nl"],
    "min_child_samples":       best_reg_params["mcs"],
    "feature_fraction":        best_reg_params["ff"],
    "bagging_fraction":        best_reg_params["bf"],
    "bagging_freq":            1,
    "reg_alpha":               best_reg_params["ra"],
    "reg_lambda":              best_reg_params["rl"],
    "min_gain_to_split":       best_reg_params["mgs"],
}

dtrain_reg = lgb.Dataset(X_train, label=y_train, free_raw_data=False)
dval_reg   = lgb.Dataset(X_val,   label=y_val,   free_raw_data=False, reference=dtrain_reg)

# Phase 1 — LR normal
print("\n🔍 Phase 1 — LR principal...")
m1 = lgb.train(
    {**reg_base, "learning_rate": best_reg_params["lr"]},
    dtrain_reg, num_boost_round=3000,
    valid_sets=[dtrain_reg, dval_reg], valid_names=["train", "val"],
    feval=wape_lgb,
    callbacks=[lgb.early_stopping(80, min_delta=1e-4), lgb.log_evaluation(200)],
)
w1 = wape(np.clip(np.round(m1.predict(X_val)), 0, None), y_val.values)
print(f"  WAPE val phase 1 : {w1:.4f} (iter={m1.best_iteration})")

# Phase 2 — LR réduit (affinage)
print("🎯 Phase 2 — LR réduit...")
m2 = lgb.train(
    {**reg_base, "learning_rate": best_reg_params["lr"] * 0.2,
     "min_child_samples": max(5, best_reg_params["mcs"] // 2),
     "reg_alpha": best_reg_params["ra"] * 0.3,
     "reg_lambda": best_reg_params["rl"] * 0.3},
    dtrain_reg, num_boost_round=4000,
    valid_sets=[dtrain_reg, dval_reg], valid_names=["train", "val"],
    feval=wape_lgb, init_model=m1,
    callbacks=[lgb.early_stopping(120, min_delta=1e-5), lgb.log_evaluation(200)],
)
w2 = wape(np.clip(np.round(m2.predict(X_val)), 0, None), y_val.values)
print(f"  WAPE val phase 2 : {w2:.4f} (iter={m2.best_iteration})")

# Choisir la meilleure phase
reg_model = m1 if w1 <= w2 else m2
print(f"\n🏆 Meilleur régresseur LGB : phase {'1' if w1 <= w2 else '2'} → WAPE={min(w1,w2):.4f}")

# Diagnostic overfitting
wape_tr = wape(np.clip(np.round(reg_model.predict(X_train)), 0, None), y_train.values)
wape_vl = wape(np.clip(np.round(reg_model.predict(X_val)),   0, None), y_val.values)
print(f"WAPE train : {wape_tr:.4f} | WAPE val : {wape_vl:.4f} | gap : {wape_vl - wape_tr:.4f} {'⚠️' if wape_vl - wape_tr > 0.05 else '✅'}")

### Régresseur XGBoost

In [0]:
print("=" * 60)
print("STAGE 2b — Régresseur XGBoost (diversité ensemble)")
print("=" * 60)

def objective_xgb(trial):
    params = {
        "objective":        "reg:tweedie",
        "tweedie_variance_power": trial.suggest_float("tvp", 1.0, 1.9),
        "eval_metric":      "mae",
        "verbosity":        0,
        "nthread":          -1,
        "seed":             SEED,
        "learning_rate":    trial.suggest_float("lr",  0.02, 0.15, log=True),
        "max_depth":        trial.suggest_int("md",    4,  10),
        "min_child_weight": trial.suggest_float("mcw", 1,  50),
        "subsample":        trial.suggest_float("ss",  0.5, 0.9),
        "colsample_bytree": trial.suggest_float("cb",  0.5, 0.9),
        "reg_alpha":        trial.suggest_float("ra",  1e-4, 1.0, log=True),
        "reg_lambda":       trial.suggest_float("rl",  1e-4, 5.0, log=True),
        "gamma":            trial.suggest_float("g",   0.0,  0.5),
    }
    dtrain_xgb = xgb.DMatrix(X_train, label=y_train)
    dval_xgb   = xgb.DMatrix(X_val,   label=y_val)
    m = xgb.train(
        params, dtrain_xgb,
        num_boost_round=800,
        evals=[(dval_xgb, "val")],
        early_stopping_rounds=30,
        verbose_eval=False,
    )
    preds = np.clip(np.round(m.predict(dval_xgb)), 0, None)
    return wape(preds, y_val.values)

print("⚙️  Optuna — XGBoost (50 trials)...")
study_xgb = optuna.create_study(direction="minimize", sampler=TPESampler(seed=SEED))
study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=False)
best_xgb_params = study_xgb.best_params
print(f"✅ Meilleurs params XGB : {best_xgb_params}")
print(f"   WAPE XGB val : {study_xgb.best_value:.4f}")

# ── Entraînement XGBoost final ─────────────────────────────────────────────
xgb_params = {
    "objective":              "reg:tweedie",
    "tweedie_variance_power": best_xgb_params["tvp"],
    "eval_metric":            "mae",
    "verbosity":              0,
    "nthread":                -1,
    "seed":                   SEED,
    "learning_rate":          best_xgb_params["lr"],
    "max_depth":              best_xgb_params["md"],
    "min_child_weight":       best_xgb_params["mcw"],
    "subsample":              best_xgb_params["ss"],
    "colsample_bytree":       best_xgb_params["cb"],
    "reg_alpha":              best_xgb_params["ra"],
    "reg_lambda":             best_xgb_params["rl"],
    "gamma":                  best_xgb_params["g"],
}

dtrain_xgb = xgb.DMatrix(X_train, label=y_train)
dval_xgb   = xgb.DMatrix(X_val,   label=y_val)

xgb_model = xgb.train(
    xgb_params, dtrain_xgb,
    num_boost_round=3000,
    evals=[(dtrain_xgb, "train"), (dval_xgb, "val")],
    early_stopping_rounds=80,
    verbose_eval=200,
)
xgb_preds_val = np.clip(np.round(xgb_model.predict(dval_xgb)), 0, None)
print(f"✅ WAPE XGB val : {wape(xgb_preds_val, y_val.values):.4f}")

### Post processing

In [0]:
print("=" * 60)
print("ENSEMBLE + POST-PROCESSING")
print("=" * 60)

# ── Prédictions brutes sur val ────────────────────────────────────────────
lgb_raw_val = np.clip(reg_model.predict(X_val), 0, None)
xgb_raw_val = np.clip(xgb_model.predict(dval_xgb), 0, None)

# ── N-1 (valeurs de l'année précédente) ──────────────────────────────────
n1_lookup = (
    train_pd[["semaine", "code_agence", "code_article", "quantite"]]
    .assign(
        annee   = lambda d: d["semaine"].str.split("-").str[0].astype(int),
        num_sem = lambda d: d["semaine"].str.split("-").str[1],
    )
    .assign(semaine_join = lambda d: (d["annee"] + 1).astype(str) + "-" + d["num_sem"])
    .rename(columns={"semaine_join": "semaine_target", "quantite": "quantite_n1"})
    [["semaine_target", "code_agence", "code_article", "quantite_n1"]]
)

def get_n1(rows_df, semaine_col="semaine"):
    rows = rows_df[[semaine_col, "code_agence", "code_article"]].copy()
    rows = rows.rename(columns={semaine_col: "semaine_target"})
    rows = rows.merge(n1_lookup, on=["semaine_target", "code_agence", "code_article"], how="left")
    return rows["quantite_n1"].fillna(0).values.astype(float)

val_rows = all_df.loc[val_mask, ["semaine", "code_agence", "code_article"]].copy()
n1_val   = get_n1(val_rows)

# ── Moyenne historique ────────────────────────────────────────────────────
pair_mean_lookup = (
    train_pd.groupby(["code_agence", "code_article"])["quantite"]
    .mean().reset_index().rename(columns={"quantite": "pair_mean_bl"})
)
pair_median_lookup = (
    train_pd.groupby(["code_agence", "code_article"])["quantite"]
    .median().reset_index().rename(columns={"quantite": "pair_median_bl"})
)

def get_pair_stat(rows_df, lookup, col):
    return rows_df.merge(lookup, on=["code_agence", "code_article"], how="left")[col].fillna(0).values.astype(float)

mean_val   = get_pair_stat(val_rows, pair_mean_lookup,   "pair_mean_bl")
median_val = get_pair_stat(val_rows, pair_median_lookup, "pair_median_bl")

# ── Stacking Ridge avec contrainte positive ────────────────────────────────
# Minimiser directement le WAPE sur la validation
stack_X = np.column_stack([lgb_raw_val, xgb_raw_val, n1_val, mean_val, median_val])
stacker  = Ridge(alpha=0.5, positive=True, fit_intercept=False)
stacker.fit(stack_X, y_val.values)

coef = stacker.coef_
print(f"Poids Ridge :")
print(f"  LGB={coef[0]:.3f} | XGB={coef[1]:.3f} | N-1={coef[2]:.3f} | Mean={coef[3]:.3f} | Median={coef[4]:.3f}")

blend_val_raw  = stacker.predict(stack_X)
blend_val_raw  = np.clip(blend_val_raw, 0, None)

# ── Appliquer le classificateur zéro ─────────────────────────────────────
clf_proba_val  = clf_model.predict(X_val)

# Seuil adaptatif : au lieu d'un seuil global, on utilise zero_rate_global
# pour ajuster le seuil par paire (produits très intermittents → seuil plus bas)
zero_rate_val  = all_df.loc[val_mask, "zero_rate_global"].fillna(0).values

# Seuil dynamique : plus le produit est souvent à zéro, plus on est agressif
adaptive_thresh = np.clip(best_thresh - 0.15 * zero_rate_val, 0.15, 0.75)
is_zero_pred    = clf_proba_val > adaptive_thresh

blend_val = np.where(is_zero_pred, 0.0, blend_val_raw)
blend_val = np.clip(np.round(blend_val), 0, None).astype(int)

# ── Post-processing : forcer zéro si zero_rate_global très élevé ──────────
# Produits quasi-toujours absents → prédire 0 systématiquement
high_zero_mask = zero_rate_val > 0.97
blend_val[high_zero_mask] = 0

print(f"\nWAPE val — LGB seul     : {wape(np.round(lgb_raw_val), y_val.values):.4f}")
print(f"WAPE val — XGB seul     : {wape(np.round(xgb_raw_val), y_val.values):.4f}")
print(f"WAPE val — blend brut   : {wape(np.round(blend_val_raw), y_val.values):.4f}")
print(f"WAPE val — + clf zéro   : {wape(blend_val, y_val.values):.4f}")
print()
print("Benchmarks :")
print("  Baseline N-1     → ~1.387")
print("  Ancien modèle    → ~1.10")
print("  Version LR dyn.  → ~0.85")
print("  → Objectif       < 0.75")

# Feature importance LGB
feat_imp = pd.DataFrame({
    "feature":    FEATURES,
    "importance": reg_model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False)
print("\nTop 20 features :")
display(feat_imp.head(20))

### 5. Validation & Score

In [0]:
print("=" * 60)
print("RETRAIN FINAL sur TRAIN + VAL → PRÉDICTIONS TEST")
print("=" * 60)

# ── Retrain LGB ───────────────────────────────────────────────────────────
print("⏳ Retrain LGB final...")
dtrain_full = lgb.Dataset(X_retrain, label=y_retrain, free_raw_data=False)

reg_model_final = lgb.train(
    {**reg_base,
     "learning_rate": best_reg_params["lr"] * 0.8},
    dtrain_full,
    num_boost_round=int(reg_model.best_iteration * 1.15),
    callbacks=[lgb.log_evaluation(-1)],
)
print(f"✅ LGB final : {reg_model_final.num_trees()} arbres")

# ── Retrain XGB ───────────────────────────────────────────────────────────
print("⏳ Retrain XGB final...")
dtrain_xgb_full = xgb.DMatrix(X_retrain, label=y_retrain)

xgb_model_final = xgb.train(
    {**xgb_params, "learning_rate": xgb_params["learning_rate"] * 0.8},
    dtrain_xgb_full,
    num_boost_round=int(xgb_model.best_ntree_limit * 1.15),
    verbose_eval=False,
)
print("✅ XGB final OK")

# ── Prédictions test ──────────────────────────────────────────────────────
lgb_raw_test = np.clip(reg_model_final.predict(X_test), 0, None)
xgb_raw_test = np.clip(xgb_model_final.predict(xgb.DMatrix(X_test)), 0, None)

test_rows = all_df.loc[test_mask, ["semaine", "code_agence", "code_article"]].copy()
n1_test   = get_n1(test_rows)
mean_test  = get_pair_stat(test_rows, pair_mean_lookup,   "pair_mean_bl")
median_test= get_pair_stat(test_rows, pair_median_lookup, "pair_median_bl")

stack_X_test  = np.column_stack([lgb_raw_test, xgb_raw_test, n1_test, mean_test, median_test])
blend_test_raw = np.clip(stacker.predict(stack_X_test), 0, None)

# Classificateur zéro final
clf_proba_test  = clf_model_final.predict(X_test)
zero_rate_test  = all_df.loc[test_mask, "zero_rate_global"].fillna(0).values
adaptive_thresh_test = np.clip(best_thresh - 0.15 * zero_rate_test, 0.15, 0.75)

is_zero_test   = clf_proba_test > adaptive_thresh_test
blend_test     = np.where(is_zero_test, 0.0, blend_test_raw)
blend_test     = np.clip(np.round(blend_test), 0, None).astype(int)

# Forcer zéro sur produits quasi-inexistants
high_zero_test = zero_rate_test > 0.97
blend_test[high_zero_test] = 0

test_rows["quantite"] = blend_test

print(f"✅ Prédictions générées : {len(test_rows):,} (attendu : 272 344)")
print(f"   Zéros prédits : {(blend_test == 0).mean():.1%}")
print(f"   Valeur max    : {blend_test.max()}")
print(f"   Valeur moy    : {blend_test.mean():.2f}")

### 6. Génération & Sauvegarde des Prédictions

In [0]:
predictions_spark = (
    spark.createDataFrame(
        test_rows[["semaine", "code_agence", "code_article", "quantite"]]
    )
    .withColumn("code_agence",  F.col("code_agence").cast(LongType()))
    .withColumn("code_article", F.col("code_article").cast(LongType()))
    .withColumn("quantite",     F.col("quantite").cast(LongType()))
)

predictions_spark.write.mode("overwrite").saveAsTable(TABLE_PREDICTIONS)
print(f"✅ Prédictions sauvegardées dans : {TABLE_PREDICTIONS}")

# Vérification rapide
check = spark.table(TABLE_PREDICTIONS)
print(f"   Lignes en base  : {check.count():,}")
print(f"   Nulls quantite  : {check.filter(F.col('quantite').isNull()).count()}")
print(f"   Valeur max      : {check.agg(F.max('quantite')).collect()[0][0]}")